# Java Security Model Benchmark — Pre vs Post Fine-tuning

Compares a base model against the fine-tuned model on **10 classic Java vulnerability scenarios**.  
Uses **natural developer prompts** — no system prompt, no special formatting — just how a developer would ask for help.

## Workflow
| Step | Command | Description |
|---|---|---|
| 1 | Run Section A | Benchmark base model → `results/base.json` |
| 2 | Run Section B | Benchmark fine-tuned model → `results/finetuned.json` |
| 3 | Run Section C | Compare both files → report |

**Scoring (0-10 per test case):**
- +1 any code block present
- +1 specifically a `java` fenced block  
- +3 vulnerable pattern removed from the output code
- +3 safe API present in the fix (capped at 3)
- +2 vulnerability type mentioned (model awareness)

### Installation

In [ ]:
%%capture
# Only needed for local mode
# !pip install transformers accelerate bitsandbytes

### Config

In [ ]:
import json, os, re, time, urllib.error, urllib.request
from datetime import datetime
from pathlib import Path

# ─── Edit these ──────────────────────────────────────────────────────────────
VLLM_URL         = os.environ.get("VLLM_BASE_URL", "http://localhost:8000")
BASE_MODEL_NAME  = "qwen3-8b"                          # name reported by vLLM server
FT_MODEL_PATH    = "output/qwen3-java-vuln-grpo_merged_16bit"  # after save/export

BASE_OUTPUT      = "results/base_model.json"
FT_OUTPUT        = "results/finetuned.json"
REPORT_OUTPUT    = "results/comparison_report.txt"

INFERENCE_MODE   = "api"    # 'api' = vLLM server  |  'local' = load model directly
MAX_TOKENS       = 1024
TEMPERATURE      = 0.3
API_KEY          = os.environ.get("VLLM_API_KEY", "")
# ─────────────────────────────────────────────────────────────────────────────

Path("results").mkdir(exist_ok=True)
print(f"vLLM URL:    {VLLM_URL}")
print(f"Base model:  {BASE_MODEL_NAME}")
print(f"FT model:    {FT_MODEL_PATH}")
print(f"Mode:        {INFERENCE_MODE}")

### Test Cases — 10 Java Vulnerability Scenarios

Each prompt simulates a real developer asking for help.  
No system prompt, no "Security Vulnerability Report" header — just natural language.

In [ ]:
TEST_CASES = [
    {
        "id": "TC-001", "cwe_id": "CWE-089", "vuln_type": "SQL Injection",
        "description": "String concatenation in JDBC query",
        "user_prompt": (
            "Hey, I wrote this Java method to look up users by email. "
            "My security scanner keeps flagging it but I'm not sure what's wrong. "
            "Can you take a look and fix it?\n\n"
            "```java\npublic User findUserByEmail(String email) {\n"
            "    String sql = \"SELECT * FROM users WHERE email = '\" + email + \"'\";\n"
            "    return jdbcTemplate.queryForObject(sql, new UserRowMapper());\n}\n```"
        ),
        "safe_apis": ["PreparedStatement", r"setString\s*\(", r"\?"],
        "vuln_patterns": [r'"\s*\+\s*email\s*\+\s*"', r"email\s*\+"],
    },
    {
        "id": "TC-002", "cwe_id": "CWE-022", "vuln_type": "Path Traversal",
        "description": "Unsanitised filename parameter used for file access",
        "user_prompt": (
            "I'm building a file download endpoint. It works fine, "
            "but my tech lead says there's a path traversal issue. What's wrong and how do I fix it?\n\n"
            "```java\npublic void downloadFile(HttpServletRequest req, HttpServletResponse resp)\n"
            "        throws IOException {\n"
            "    String filename = req.getParameter(\"file\");\n"
            "    File file = new File(\"/var/uploads/\" + filename);\n"
            "    Files.copy(file.toPath(), resp.getOutputStream());\n}\n```"
        ),
        "safe_apis": [r"normalize\(\)", r"Paths\.get\(", r"startsWith\(", r"toRealPath"],
        "vuln_patterns": [r'new File\(.*\+\s*filename', r'"/.*"\s*\+\s*filename'],
    },
    {
        "id": "TC-003", "cwe_id": "CWE-078", "vuln_type": "OS Command Injection",
        "description": "User input passed to Runtime.exec()",
        "user_prompt": (
            "This utility pings a host to check if it's reachable. "
            "A coworker said it's dangerous but I don't see why. Can you explain and fix it?\n\n"
            "```java\npublic String pingHost(String hostname) throws IOException {\n"
            "    Process proc = Runtime.getRuntime().exec(\"ping -c 1 \" + hostname);\n"
            "    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream()))) {\n"
            "        return br.lines().collect(Collectors.joining(\"\\n\"));\n    }\n}\n```"
        ),
        "safe_apis": ["ProcessBuilder", r"Arrays\.asList\(", r"allowlist", r"whitelist", r"matches\("],
        "vuln_patterns": [r"exec\(.*\+\s*hostname", r"getRuntime\(\)\.exec\("],
    },
    {
        "id": "TC-004", "cwe_id": "CWE-611", "vuln_type": "XML External Entity (XXE)",
        "description": "Default DocumentBuilder allows external entity processing",
        "user_prompt": (
            "I parse XML from user submissions. OWASP says DocumentBuilderFactory can be exploited. "
            "Can you make it safe?\n\n"
            "```java\npublic Document parseXml(InputStream xmlInput) throws Exception {\n"
            "    DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n"
            "    DocumentBuilder builder = factory.newDocumentBuilder();\n"
            "    return builder.parse(xmlInput);\n}\n```"
        ),
        "safe_apis": [r"setFeature\(", r"external-general-entities", r"disallow-doctype-decl"],
        "vuln_patterns": [r"newDocumentBuilder\(\)\s*;\s*\n\s*return builder\.parse"],
    },
    {
        "id": "TC-005", "cwe_id": "CWE-502", "vuln_type": "Insecure Deserialization",
        "description": "ObjectInputStream deserialises untrusted data",
        "user_prompt": (
            "Our app deserialises objects from the network. The pentest report says this is critical. "
            "How do I fix it?\n\n"
            "```java\npublic Object deserializeFromStream(InputStream is) throws Exception {\n"
            "    ObjectInputStream ois = new ObjectInputStream(is);\n"
            "    return ois.readObject();\n}\n```"
        ),
        "safe_apis": [r"ValidatingObjectInputStream", r"ObjectInputFilter", r"createFilter", r"Jackson", r"Gson"],
        "vuln_patterns": [r"new ObjectInputStream\(is\)", r"new ObjectInputStream\("],
    },
    {
        "id": "TC-006", "cwe_id": "CWE-327", "vuln_type": "Weak Cryptographic Algorithm",
        "description": "MD5 used for password hashing",
        "user_prompt": (
            "I'm hashing passwords before storing them. I heard MD5 is insecure — "
            "what should I use instead? Can you update this code?\n\n"
            "```java\npublic String hashPassword(String password) throws NoSuchAlgorithmException {\n"
            "    MessageDigest md = MessageDigest.getInstance(\"MD5\");\n"
            "    byte[] hash = md.digest(password.getBytes());\n"
            "    return Base64.getEncoder().encodeToString(hash);\n}\n```"
        ),
        "safe_apis": [r"BCrypt", r"Argon2", r"PBKDF2WithHmacSHA", r"SCrypt", r"bcrypt"],
        "vuln_patterns": [r'getInstance\("MD5"\)', r'getInstance\("SHA-1"\)'],
    },
    {
        "id": "TC-007", "cwe_id": "CWE-330", "vuln_type": "Use of Insufficiently Random Values",
        "description": "java.util.Random used for security tokens",
        "user_prompt": (
            "I generate password reset tokens like this but code review says I should use \"secure\" random. "
            "What's the difference and how do I fix it?\n\n"
            "```java\npublic String generateResetToken() {\n"
            "    Random random = new Random();\n"
            "    byte[] bytes = new byte[32];\n"
            "    random.nextBytes(bytes);\n"
            "    return Base64.getUrlEncoder().encodeToString(bytes);\n}\n```"
        ),
        "safe_apis": [r"SecureRandom", r"new SecureRandom"],
        "vuln_patterns": [r"new Random\(\)", r"new java\.util\.Random"],
    },
    {
        "id": "TC-008", "cwe_id": "CWE-090", "vuln_type": "LDAP Injection",
        "description": "Unsanitised username in LDAP search filter",
        "user_prompt": (
            "We use Active Directory for auth. Can you check if this lookup has any security issues?\n\n"
            "```java\npublic boolean authenticateUser(String username, String password)\n"
            "        throws NamingException {\n"
            "    String filter = \"(&(objectClass=user)(sAMAccountName=\" + username + \"))\";\n"
            "    NamingEnumeration<?> results = ctx.search(\"dc=company,dc=com\", filter, controls);\n"
            "    return results.hasMore();\n}\n```"
        ),
        "safe_apis": [r"escapeLDAPSearchFilter", r"encodeForLDAP", r"ESAPI", r"replace\(", r"whitelist"],
        "vuln_patterns": [r'filter.*\+.*username', r'sAMAccountName="\s*\+'],
    },
    {
        "id": "TC-009", "cwe_id": "CWE-918", "vuln_type": "Server-Side Request Forgery (SSRF)",
        "description": "User-controlled URL used for outbound HTTP",
        "user_prompt": (
            "I'm building a webhook feature where we call back to a URL the user provides. "
            "Security flagged it as SSRF. How do I fix this?\n\n"
            "```java\npublic String callWebhook(String webhookUrl, String payload) throws IOException {\n"
            "    URL url = new URL(webhookUrl);\n"
            "    HttpURLConnection conn = (HttpURLConnection) url.openConnection();\n"
            "    conn.setRequestMethod(\"POST\");\n"
            "    conn.setDoOutput(true);\n"
            "    try (OutputStream os = conn.getOutputStream()) { os.write(payload.getBytes()); }\n"
            "    return String.valueOf(conn.getResponseCode());\n}\n```"
        ),
        "safe_apis": [r"allowlist", r"whitelist", r"InetAddress", r"getHost\(\)", r"validate"],
        "vuln_patterns": [r"new URL\(webhookUrl\)", r"URL\(.*webhookUrl"],
    },
    {
        "id": "TC-010", "cwe_id": "CWE-079", "vuln_type": "Cross-Site Scripting (XSS)",
        "description": "Unsanitised user input reflected in HTML response",
        "user_prompt": (
            "My search page echoes the query back to the user. My colleague mentioned XSS risk. "
            "Is there a problem and how do I fix it?\n\n"
            "```java\npublic void handleSearch(HttpServletRequest req, HttpServletResponse resp)\n"
            "        throws IOException {\n"
            "    String query = req.getParameter(\"q\");\n"
            "    resp.setContentType(\"text/html\");\n"
            "    PrintWriter out = resp.getWriter();\n"
            "    out.println(\"<h2>Results for: \" + query + \"</h2>\");\n}\n```"
        ),
        "safe_apis": [r"HtmlUtils\.htmlEscape", r"StringEscapeUtils", r"escapeHtml", r"ESAPI", r"Encode\.forHtml"],
        "vuln_patterns": [r'println\(.*\+\s*query', r'".*"\s*\+\s*query\s*\+'],
    },
]

print(f"{len(TEST_CASES)} test cases loaded.")
for tc in TEST_CASES:
    print(f"  {tc['id']}  {tc['cwe_id']:<10} {tc['vuln_type']}")

### Scoring & Inference Helpers

In [ ]:
MATCH_CODE = re.compile(r'```(\w+)?\s*([\s\S]+?)```', re.DOTALL)
MATCH_JAVA = re.compile(r'```java\s*([\s\S]+?)```', re.DOTALL)

def score_response(tc: dict, response: str) -> dict:
    java_m = MATCH_JAVA.search(response)
    code_m = MATCH_CODE.search(response)
    java_code = (java_m.group(2) if java_m else (code_m.group(2) if code_m else response)).strip()

    score = 0
    d = {}

    d["has_code_block"] = bool(code_m)
    if d["has_code_block"]: score += 1

    d["has_java_block"] = bool(java_m)
    if d["has_java_block"]: score += 1

    vuln_hits = sum(1 for p in tc["vuln_patterns"] if re.search(p, java_code, re.IGNORECASE))
    d["vuln_patterns_still_present"] = vuln_hits
    d["vuln_patterns_removed"]       = len(tc["vuln_patterns"]) - vuln_hits
    if vuln_hits == 0 and java_code: score += 3
    elif vuln_hits < len(tc["vuln_patterns"]): score += 1

    safe_hits = sum(1 for p in tc["safe_apis"] if re.search(p, java_code, re.IGNORECASE))
    d["safe_apis_found"] = safe_hits
    score += min(safe_hits, 3)

    kws = [w for w in tc["vuln_type"].lower().split() if len(w) > 3]
    d["vulnerability_mentioned"] = any(kw in response.lower() for kw in kws)
    if d["vulnerability_mentioned"]: score += 2

    d["score"] = min(score, 10)
    d["max_score"] = 10
    return d


def call_api(model: str, prompt: str) -> tuple:
    """Call vLLM API with NO system prompt. Returns (text, latency_ms, n_tokens)."""
    url     = f"{VLLM_URL.rstrip('/')}/v1/chat/completions"
    payload = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": TEMPERATURE,
        "max_tokens":  MAX_TOKENS,
    }).encode()
    headers = {"Content-Type": "application/json"}
    if API_KEY: headers["Authorization"] = f"Bearer {API_KEY}"
    req = urllib.request.Request(url, data=payload, headers=headers, method="POST")
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=300) as resp:
        ms   = (time.time() - t0) * 1000
        data = json.loads(resp.read().decode())
        text = data["choices"][0]["message"]["content"]
        n    = data.get("usage", {}).get("completion_tokens", len(text.split()))
        return text, ms, n


def call_local(model_path: str, prompt: str) -> tuple:
    """Load model locally and run inference. Returns (text, latency_ms, n_tokens)."""
    import gc, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"  Loading {model_path} locally...")
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16,
                                                device_map="auto", trust_remote_code=True)
    mdl.eval()
    try:
        text_in = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                           tokenize=False, add_generation_prompt=True)
    except Exception:
        text_in = prompt
    inputs = tok(text_in, return_tensors="pt").to(mdl.device)
    n_in   = inputs["input_ids"].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=MAX_TOKENS, temperature=TEMPERATURE,
                           do_sample=True, pad_token_id=tok.eos_token_id)
    ms  = (time.time() - t0) * 1000
    n   = out.shape[1] - n_in
    txt = tok.decode(out[0][n_in:], skip_special_tokens=True)
    del mdl, tok; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return txt, ms, n


def infer(model_name_or_path: str, prompt: str) -> tuple:
    if INFERENCE_MODE == "local":
        return call_local(model_name_or_path, prompt)
    return call_api(model_name_or_path, prompt)


print("Scoring and inference helpers ready.")

---
## Section A — Benchmark Base Model

In [ ]:
def run_benchmark(model_id: str, output_path: str):
    results = []
    print(f"\nBenchmarking: {model_id}")
    print(f"Output:       {output_path}")
    print("=" * 65)

    for tc in TEST_CASES:
        print(f"\n  [{tc['id']}] {tc['cwe_id']} — {tc['vuln_type']}")
        try:
            response, latency_ms, n_tok = infer(model_id, tc["user_prompt"])
            tps     = n_tok / (latency_ms / 1000) if latency_ms > 0 else 0
            scoring = score_response(tc, response)
            print(f"         Score: {scoring['score']:2d}/10 | "
                  f"Latency: {latency_ms:.0f}ms | Throughput: {tps:.1f} tok/s")
            print(f"         Safe APIs: {scoring['safe_apis_found']} | "
                  f"Vuln removed: {scoring['vuln_patterns_removed']}/{len(tc['vuln_patterns'])}")
            results.append({
                "test_id": tc["id"], "cwe_id": tc["cwe_id"],
                "vuln_type": tc["vuln_type"], "description": tc["description"],
                "input_prompt": tc["user_prompt"], "model_output": response,
                "latency_ms": round(latency_ms, 1), "tokens_generated": n_tok,
                "tokens_per_sec": round(tps, 1),
                "scoring": scoring,
                "timestamp": datetime.now().isoformat(), "model_name": model_id,
            })
        except Exception as e:
            print(f"         [ERROR] {e}")
            results.append({
                "test_id": tc["id"], "cwe_id": tc["cwe_id"],
                "vuln_type": tc["vuln_type"], "description": tc["description"],
                "input_prompt": tc["user_prompt"], "model_output": f"ERROR: {e}",
                "latency_ms": -1, "tokens_generated": 0, "tokens_per_sec": 0,
                "scoring": {"score": 0, "max_score": 10},
                "timestamp": datetime.now().isoformat(), "model_name": model_id,
            })

    avg_s   = sum(r["scoring"]["score"] for r in results) / len(results)
    valid   = [r for r in results if r["latency_ms"] > 0]
    avg_lat = sum(r["latency_ms"] for r in valid) / max(len(valid), 1)
    avg_tps = sum(r["tokens_per_sec"] for r in valid) / max(len(valid), 1)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump({"model": model_id, "mode": INFERENCE_MODE,
                   "timestamp": datetime.now().isoformat(),
                   "avg_score": round(avg_s, 2),
                   "avg_latency_ms": round(avg_lat, 1),
                   "avg_tokens_per_sec": round(avg_tps, 1),
                   "results": results}, f, indent=2)
    print(f"\n{'='*65}")
    print(f"  Avg score:      {avg_s:.1f}/10")
    print(f"  Avg latency:    {avg_lat:.0f} ms")
    print(f"  Avg throughput: {avg_tps:.1f} tok/s")
    print(f"  Saved to:       {output_path}")
    return results


base_results = run_benchmark(BASE_MODEL_NAME, BASE_OUTPUT)

In [ ]:
# Show first response to sanity-check
r = base_results[0]
print(f"[{r['test_id']}] {r['cwe_id']} — Score: {r['scoring']['score']}/10")
print("─" * 60)
print(r["model_output"][:1000])

---
## Section B — Benchmark Fine-tuned Model

Make sure the fine-tuned model is exported and either:
- Loaded into the vLLM server (`vllm serve output/qwen3-java-vuln-grpo_merged_16bit`), **or**
- Set `INFERENCE_MODE = "local"` in Config cell

In [ ]:
ft_results = run_benchmark(FT_MODEL_PATH, FT_OUTPUT)

In [ ]:
# Show same test case for the fine-tuned model
r = ft_results[0]
print(f"[{r['test_id']}] {r['cwe_id']} — Score: {r['scoring']['score']}/10")
print("─" * 60)
print(r["model_output"][:1000])

---
## Section C — Compare Results

In [ ]:
with open(BASE_OUTPUT) as f:  data_a = json.load(f)
with open(FT_OUTPUT)   as f:  data_b = json.load(f)

r_a = {r["test_id"]: r for r in data_a["results"]}
r_b = {r["test_id"]: r for r in data_b["results"]}
ids = sorted(set(r_a) | set(r_b))

print(f"{'='*65}")
print(f"  Base model:        {data_a['model']}")
print(f"  Fine-tuned model:  {data_b['model']}")
print(f"{'='*65}")
print(f"  {'ID':<8} {'Vulnerability':<30} {'Base':>5} {'FT':>5} {'Δ':>6}")
print("  " + "-" * 55)

improved = regressed = unchanged = 0
for tid in ids:
    ra = r_a.get(tid); rb = r_b.get(tid)
    if not ra or not rb: continue
    sa = ra["scoring"]["score"]; sb = rb["scoring"]["score"]; d = sb - sa
    if d > 0:   improved  += 1; arrow = "▲"
    elif d < 0: regressed += 1; arrow = "▼"
    else:       unchanged += 1; arrow = "─"
    print(f"  {tid:<8} {ra['vuln_type'][:28]:<30} {sa:>5.1f} {sb:>5.1f} {d:>+5.1f}{arrow}")

avg_a = data_a.get("avg_score", 0); avg_b = data_b.get("avg_score", 0)
print("  " + "-" * 55)
print(f"  {'AVERAGE':<39} {avg_a:>5.1f} {avg_b:>5.1f} {avg_b - avg_a:>+5.1f}")
print(f"\n  Improved: {improved}  Regressed: {regressed}  Unchanged: {unchanged}")
print(f"  Avg latency  — base: {data_a['avg_latency_ms']:.0f}ms  FT: {data_b['avg_latency_ms']:.0f}ms")
print(f"  Avg tok/s    — base: {data_a['avg_tokens_per_sec']:.1f}    FT: {data_b['avg_tokens_per_sec']:.1f}")

delta = avg_b - avg_a
print("\n  Verdict:")
if delta >= 3:    print("    ✅ SIGNIFICANT IMPROVEMENT")
elif delta >= 1:  print("    ✅ MODERATE IMPROVEMENT")
elif delta >= 0:  print("    ⚠️  MARGINAL — consider more training steps or data")
else:             print("    ❌ REGRESSION — review reward functions and training")

In [ ]:
# Side-by-side output for a specific test case
VIEW_ID = "TC-001"   # ← change this to view a different test case

ra = r_a[VIEW_ID]; rb = r_b[VIEW_ID]
print(f"[{VIEW_ID}] {ra['cwe_id']} — {ra['vuln_type']}")
print(f"Base score: {ra['scoring']['score']}/10   |   FT score: {rb['scoring']['score']}/10")

print("\n" + "─"*30 + " BASE MODEL " + "─"*30)
print(ra["model_output"])

print("\n" + "─"*28 + " FINE-TUNED MODEL " + "─"*28)
print(rb["model_output"])

In [ ]:
# Save full comparison report
import subprocess
result = subprocess.run(
    ["python", "modelcompare-pre-post.py",
     "--compare", BASE_OUTPUT, FT_OUTPUT,
     "--report",  REPORT_OUTPUT],
    capture_output=True, text=True, cwd="."
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print(f"\nFull report saved to: {REPORT_OUTPUT}")

---
### Visualisation — Score per Test Case

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    ids_plot  = [r["test_id"]   for r in data_a["results"]]
    scores_a  = [r["scoring"]["score"] for r in data_a["results"]]
    scores_b  = [r["scoring"]["score"] for r in data_b["results"]]

    x  = np.arange(len(ids_plot))
    w  = 0.35
    fig, ax = plt.subplots(figsize=(14, 5))
    bars_a = ax.bar(x - w/2, scores_a, w, label=f"Base ({data_a['model'][:20]})", color="#4C72B0")
    bars_b = ax.bar(x + w/2, scores_b, w, label=f"FT   ({data_b['model'][:20]})", color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels(ids_plot, rotation=30, ha="right")
    ax.set_ylabel("Score (0–10)")
    ax.set_ylim(0, 11)
    ax.set_title("Security Benchmark: Base vs Fine-tuned Model")
    ax.legend()
    ax.bar_label(bars_a, padding=2, fontsize=8)
    ax.bar_label(bars_b, padding=2, fontsize=8)
    plt.tight_layout()
    plt.savefig("results/comparison_chart.png", dpi=150)
    plt.show()
    print("Chart saved to results/comparison_chart.png")
except ImportError:
    print("matplotlib not installed — skipping chart. Install with: pip install matplotlib")